# Notebook 2 — Feature Engineering

**Goal:** Derive features that improve anomaly detection beyond the raw cgminer fields.

**Approach:**
1. **Domain features** — ratios and rates that have physical meaning (efficiency, temp differential, etc.)
2. **Rolling-window features** — short-term volatility captures sudden changes
3. **Cumulative-to-delta** — cumulative counters (Hardware Errors) become rate-of-change
4. **Feature selection** — measure which features actually help the model using mutual information vs the labels

**Note on unsupervised use:** the model itself is unsupervised. We only use labels here to *validate* feature engineering choices.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.feature_selection import mutual_info_classif

sns.set_style('whitegrid')
DATA_DIR = Path('data')
df = pd.read_csv(DATA_DIR / 'miner_telemetry_synthetic.csv', parse_dates=['timestamp'])
print(f'Loaded {len(df)} samples')

## 1. Domain features

Derived features that an experienced miner operator would compute by hand. They have physical meaning and capture multivariate relationships in a single number — easier for the model to learn.

| Feature | Formula | Physical meaning |
|---|---|---|
| `hash_efficiency` | GHS 5s / chain_power | Hashes per watt — drops if chips are unhealthy |
| `temp_differential` | temp_chip - temp_pcb | Temperature rise across the board — large delta = poor heat transfer |
| `chain_rate_imbalance` | std(chain_rate1..4) / mean | How unevenly the boards are working |
| `voltage_imbalance` | std(voltage1..4) | Power supply asymmetry |
| `total_active_chips` | sum(chain_acn1..4) | Total chip count, easier to monitor than per-board |
| `hashrate_per_chip` | GHS 5s / total_active_chips | Per-chip efficiency |
| `fan_avg` | mean(fan1, fan2) | Smoothed cooling effort |
| `temp_max_minus_avg` | temp_max - mean(temp2_1..4) | Hot-spot indicator |

In [ ]:
def add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['hash_efficiency']     = df['GHS 5s'] / df['chain_power'].replace(0, np.nan)
    df['temp_differential']   = (df[['temp2_1','temp2_2','temp2_3','temp2_4']].mean(axis=1)
                                  - df[['temp1','temp2','temp3','temp4']].mean(axis=1))
    df['chain_rate_imbalance']= df[['chain_rate1','chain_rate2','chain_rate3','chain_rate4']].std(axis=1) / \
                                df[['chain_rate1','chain_rate2','chain_rate3','chain_rate4']].mean(axis=1)
    df['voltage_imbalance']   = df[['voltage1','voltage2','voltage3','voltage4']].std(axis=1)
    df['total_active_chips']  = df[['chain_acn1','chain_acn2','chain_acn3','chain_acn4']].sum(axis=1)
    df['hashrate_per_chip']   = df['GHS 5s'] / df['total_active_chips'].replace(0, np.nan)
    df['fan_avg']             = (df['fan1'] + df['fan2']) / 2
    df['temp_max_minus_avg']  = df['temp_max'] - df[['temp2_1','temp2_2','temp2_3','temp2_4']].mean(axis=1)
    return df

df_eng = add_domain_features(df)
domain_feats = ['hash_efficiency','temp_differential','chain_rate_imbalance',
                'voltage_imbalance','total_active_chips','hashrate_per_chip',
                'fan_avg','temp_max_minus_avg']
df_eng[domain_feats].describe().round(4)

**Visual check:** does each domain feature actually separate normal from anomaly?

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, f in zip(axes.flatten(), domain_feats):
    sns.kdeplot(data=df_eng, x=f, hue='anomaly_label', ax=ax,
                palette={'normal':'#22c55e','anomaly':'#ef4444'},
                fill=True, alpha=0.4, common_norm=False)
    ax.set_title(f, fontsize=10); ax.legend([], frameon=False)
plt.tight_layout(); plt.show()

Strong separation on `hash_efficiency`, `chain_rate_imbalance`, `voltage_imbalance`, `total_active_chips`, `hashrate_per_chip`, `temp_max_minus_avg`. These are the highest-value derived features.

## 2. Rolling-window features

Time-series anomalies often show as sudden changes (rate-of-change) or short-term volatility. We compute rolling stats over 5-sample windows (~2.5 minutes).

In [ ]:
def add_rolling_features(df: pd.DataFrame, window: int = 5) -> pd.DataFrame:
    df = df.copy()
    key_feats = ['GHS 5s', 'temp_max', 'fan1', 'fan2', 'voltage1']
    for f in key_feats:
        df[f'{f}_roll_std']  = df[f].rolling(window, min_periods=1).std()
        df[f'{f}_roll_mean'] = df[f].rolling(window, min_periods=1).mean()
        # rate of change — current vs window mean
        df[f'{f}_roc']       = (df[f] - df[f'{f}_roll_mean']) / df[f'{f}_roll_mean'].replace(0, np.nan)
    return df

df_eng = add_rolling_features(df_eng, window=5)
rolling_cols = [c for c in df_eng.columns if '_roll_' in c or '_roc' in c]
print(f'Added {len(rolling_cols)} rolling features')

## 3. Mutual information ranking

We use mutual information (MI) between each feature and the anomaly label as a proxy for feature importance.

**Why MI instead of correlation?** MI captures non-linear relationships. For our anomalies (chip drop, voltage swings) the relationship to the label is highly non-linear.

**Why use labels here?** Only to *evaluate* engineering decisions. The actual model trains unsupervised. This is the standard methodology for unsupervised feature selection — you score features against synthetic labels then deploy the unsupervised model with those features.

In [ ]:
y = (df_eng['anomaly_label'] == 'anomaly').astype(int)
all_feats = [c for c in df_eng.columns if c not in ('timestamp','anomaly_label','anomaly_type')]
X = df_eng[all_feats].fillna(0)
# Drop constant columns (would have MI = 0 anyway, slows down computation)
X = X.loc[:, X.std() > 0]

mi = mutual_info_classif(X, y, random_state=42, n_neighbors=3)
mi_series = pd.Series(mi, index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(11, 11))
mi_series.plot.barh(ax=ax, color='#06b6d4')
ax.invert_yaxis(); ax.set_xlabel('Mutual information vs anomaly label')
ax.set_title('Feature importance ranking (MI score)')
plt.tight_layout(); plt.show()
print('Top 15 features by MI:')
print(mi_series.head(15).round(4))

## 4. Feature selection decision

Three candidate feature sets:

**Set A — Raw only (41 features):** the original cgminer fields.

**Set B — Raw + Domain (49 features):** raw fields plus the 8 domain features.

**Set C — Top 25 by MI:** keep only the most informative features.

We evaluate all three in notebook 3 to pick the best.

In [ ]:
raw_feats    = [c for c in df.columns if c not in ('timestamp','anomaly_label','anomaly_type')]
domain_only  = domain_feats
top_25_mi    = mi_series.head(25).index.tolist()

feature_sets = {
    'A_raw_only':    raw_feats,
    'B_raw_domain':  raw_feats + domain_only,
    'C_top25_MI':    top_25_mi,
}
for name, feats in feature_sets.items():
    print(f'{name:20} → {len(feats)} features')

# Save the engineered dataset
df_eng.to_csv(DATA_DIR / 'miner_telemetry_engineered.csv', index=False)
import json
with open(DATA_DIR / 'feature_sets.json', 'w') as f:
    json.dump(feature_sets, f, indent=2)
print('Saved engineered dataset and feature sets')

## Summary

- Added **8 domain features** with strong separating power (especially `chain_rate_imbalance`, `voltage_imbalance`, `total_active_chips`)
- Added **15 rolling-window features** capturing short-term volatility
- Mutual information identified `chain_rate_imbalance`, `total_active_chips`, `voltage_imbalance`, `chain_acn4`, `Device Rejected%` as the top discriminators
- Saved 3 candidate feature sets for model comparison

**Next:** Notebook 3 — model selection.